# Chapter 1: From words to vectors

**Theory 15 min, exercise 10 min.**

A language model can do two very different things with a piece of text. It can *continue* it, which is what chatbots do, or it can *describe* it as a list of numbers. This tutorial is entirely about the second use. By the end of this chapter you will know what those numbers are, where they come from, and how to compare them.

## Theory

### Two things a language model can give you

![Text generation versus feature extraction](images/two_uses.png)

Every modern language model is a neural network that reads text and produces internal activations. Those activations can be decoded into the next word, which gives **text generation**. Or they can simply be read out as they are, which gives **feature extraction**: a fixed-length vector of numbers that summarises the input. That vector is the **embedding**.

Feature extraction is the quieter use, but for research it has three advantages. It is deterministic, it is cheap enough to run on a laptop over thousands of texts, and it hands you a numerical object that fits into the statistics you already know.

### Meaning from context

![Firth's distributional hypothesis](images/firth.png)

The idea behind every embedding model is older than neural networks. In the 1950s the linguist J. R. Firth proposed that the meaning of a word is given by the words that tend to occur around it. *Galaxy* and *cosmos* appear in the same sort of sentences, so they mean similar things. *Galaxy* and *Switzerland* rarely do. This is the **distributional hypothesis**, and it turns meaning into something you can count.

### Counting contexts: latent semantic analysis

![Latent semantic analysis builds a word-by-context count matrix and compresses it](images/lsa.png)

The first practical implementation, in psychology no less, was **latent semantic analysis** (Landauer and Dumais, 1997). Take a large corpus, make a table with one row per word and one column per context (a document, or a window of neighbouring words), and count. Each row is already a crude vector for the word. Because the table is huge and sparse, it is re-weighted and compressed with a singular value decomposition to a few hundred dimensions. Those compressed rows are the embeddings.

### Predicting contexts: word2vec

![word2vec: a one-hot input word, a small hidden layer, and the context words as output](images/word2vec.png)

In 2013, word2vec replaced counting with prediction. A tiny neural network takes a single word as a one-hot input and is trained to predict the words around it. The network is forced through a narrow hidden layer of, say, 300 units. To predict well, it must place words with similar contexts close together in that layer.

![A good model encodes words with similar contexts similarly](images/similar_context.png)

After training, the hidden layer is the embedding. Nobody told the network that *galaxy* and *cosmos* are related. It discovered it because the two words need the same output.

### An embedding is a hidden activation

![Embeddings are the hidden activations recorded for an input](images/hidden_activations.png)

This is the general recipe, and it is the same for the transformer models you will use in the next chapter. Train a network on some language task, then throw the output layer away and keep the activations just before it. Feed in an input, record the activations, and you have its vector.

![Recording the hidden activations for several inputs gives an embedding table](images/embedding_table.png)

Doing this for many inputs yields an **embedding table**: one row per input, one column per hidden dimension. That table is a perfectly ordinary data matrix. You can correlate its rows, cluster them, reduce them to two dimensions, or use them as predictors.

### Comparing vectors: cosine similarity

![Cosine similarity between two embedding vectors](images/cosine.png)

The individual numbers in an embedding are not interpretable. The *angle* between two embeddings is. **Cosine similarity** is the cosine of that angle:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}$$

It ranges from 1 (same direction, same meaning) through 0 (unrelated) to -1 (opposite). Dividing by the vector lengths means a long, emphatic text and a short, plain text about the same topic still come out similar. It is also exactly the Pearson correlation of the two vectors when they have mean zero, which is a useful way to think about it.

![Cosine similarity for every pair of inputs gives a similarity matrix](images/cosine_matrix.png)

Computing cosine similarity for every pair of inputs gives a symmetric **similarity matrix** with ones on the diagonal. For psychologists this object should look familiar: it has the same shape as an inter-item correlation matrix, and in Chapter 2 we will compare the two directly.

```{admonition} Check your understanding
:class: note
1. Why can two words that never appear in the same sentence still get similar embeddings?
2. In word2vec, what forces the hidden layer to encode meaning rather than, say, spelling?
3. Two texts have cosine similarity 0.9. Does that mean they say the same thing, or that they are about the same thing? What is the difference?
```

## Exercise

Press the rocket at the top of the page and choose **Colab**, then run the cells in order.

### Setup

The first cell installs `sentence-transformers` on Colab and loads a small embedding model. It takes about a minute.

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q sentence-transformers

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Data is read straight from the public course repository; nothing to upload or mount.
DATA = "https://raw.githubusercontent.com/dwulff/LLM4BeSci_2025MetaRep/main/day_2/"

# The embedding model used in this chapter. 22M parameters, 384 dimensions, fast on a CPU.
MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
print("Loaded", MODEL_NAME, "with", model.get_sentence_embedding_dimension(), "dimensions")

Loaded all-MiniLM-L6-v2 with 384 dimensions


### Three sentences become three vectors

Run the cell below. Each row is one sentence, each column one of the 384 embedding dimensions.

In [2]:
sentences = [
    "I feel great this morning",
    "I am feeling very good today",
    "I am feeling terrible",
]

features = model.encode(sentences)
pd.DataFrame(features, index=sentences).round(3)

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
I feel great this morning,-0.026,-0.044,0.072,0.035,0.090,-0.050,0.019,0.071,-0.021,-0.044,...,-0.006,-0.000,-0.049,0.016,-0.028,0.017,0.065,0.017,-0.023,-0.037
I am feeling very good today,-0.044,-0.020,0.067,-0.006,0.026,-0.040,0.079,-0.010,-0.043,-0.026,...,-0.045,0.049,-0.049,0.018,-0.018,-0.010,0.043,0.018,-0.008,-0.006
I am feeling terrible,0.017,-0.058,0.033,0.002,0.052,-0.048,0.008,0.119,0.030,-0.069,...,0.039,0.003,-0.075,-0.018,-0.026,0.006,0.051,-0.010,0.030,-0.064


The raw numbers tell you nothing on their own. What matters is the geometry between rows. The cell below computes the cosine similarity between every pair of sentences.

In [3]:
sims = cosine_similarity(features)
pd.DataFrame(sims, index=sentences, columns=sentences).round(2)

,I feel great this morning,I am feeling very good today,I am feeling terrible
I feel great this morning,1.00,0.79,0.59
I am feeling very good today,0.79,1.00,0.58
I am feeling terrible,0.59,0.58,1.00


The two positive sentences are close to each other and far from the negative one, even though the second and third share more words than the first and second.

**Task 1.** Add a fourth sentence that paraphrases one of the others (for example swap *great* for *fantastic*). Rerun both cells. Where does it land?

**Task 2.** Add a sentence that shares words but not meaning (for example *"I am feeling the table"*). Does the model fall for surface overlap?

**Task 3.** Add a sentence in another language that means the same as the first one. This model was trained mostly on English. How does it cope? The multilingual sibling `paraphrase-multilingual-MiniLM-L12-v2` is one `MODEL_NAME` change away if you want to compare.